1. Бібліотеки, завантаження та підготовка 768D ембедингів

In [ ]:
import numpy as np
import pandas as pd
import os
import torch
import re
from sentence_transformers import SentenceTransformer
from scipy.linalg import polar, svd, orthogonal_procrustes
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. ВИЗНАЧЕННЯ ПРИСТРОЮ (GPU або CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Використовується пристрій: {device}")

# 2. ЗАВАНТАЖЕННЯ МОДЕЛІ (768 вимірів)
model_name = 'all-mpnet-base-v2'
print(f"Завантаження моделі {model_name}...")
model = SentenceTransformer(model_name, device=device)

# --- ДОПОМІЖНІ ФУНКЦІЇ ---

def parse_pit_votes(vote_str):
    """ Парсинг голосів анотаторів типу '(5, 0)' у бінарну мітку (позитив >= 3) """
    try:
        nums = re.findall(r'\d+', str(vote_str))
        if len(nums) >= 1:
            return 1 if int(nums[0]) >= 3 else 0
        return 0
    except:
        return 0

def clean_pit_text(text):
    """ Видалення лінгвістичних тегів /NNP, /VBZ тощо для чистої семантики """
    if pd.isna(text): return ""
    text_str = str(text)
    if '/' in text_str:
        return " ".join([word.split('/')[0] for word in text_str.split()])
    return text_str

def load_pit_dataset(filename):
    """ Завантаження PIT-2015 з локальної пам'яті Colab """
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Файл {filename} не знайдено! Перетягніть 'train.data' та 'dev.data' у вікно Files.")

    df = pd.read_csv(filename, sep='\t', header=None, on_bad_lines='skip')
    # 0: TopicID, 2: Sent1 (Тема), 3: Sent2 (Твіт), 4: Votes
    df = df.iloc[:, [0, 2, 3, 4]]
    df.columns = ['Topic_ID', 'Sent1_raw', 'Sent2_raw', 'Votes']

    df['Label'] = df['Votes'].apply(parse_pit_votes)
    df['Sent1_clean'] = df['Sent1_raw'].apply(clean_pit_text)
    df['Sent2_clean'] = df['Sent2_raw'].apply(clean_pit_text)
    return df

def encode_and_normalize(sentences):
    """ Отримання ембедингів SBERT та L2-нормалізація (проекція на гіперсферу) """
    sent_list = [str(s) for s in sentences]
    emb = model.encode(sent_list, show_progress_bar=True, convert_to_numpy=True)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    return emb / (norms + 1e-9)

# --- ВИКОНАННЯ ---

print("\nКрок 1: Завантаження та очищення текстів...")
df_train = load_pit_dataset('train.data')
df_dev = load_pit_dataset('dev.data')

print(f"У Train: {len(df_train)} рядків, з них парафраз (Label=1): {df_train['Label'].sum()}")
print(f"У Dev:   {len(df_dev)} рядків, з них парафраз (Label=1): {df_dev['Label'].sum()}")

print("\nКрок 2: Генерація ембедингів 768D...")
# X - Оригінали (Sent1_clean), Xp - Парафрази (Sent2_clean)
X_train_full = encode_and_normalize(df_train['Sent1_clean'].tolist())
Xp_train_full = encode_and_normalize(df_train['Sent2_clean'].tolist())

X_dev = encode_and_normalize(df_dev['Sent1_clean'].tolist())
Xp_dev = encode_and_normalize(df_dev['Sent2_clean'].tolist())

print("\n" + "="*40)
print("ПІДГОТОВКА ДАНИХ ЗАВЕРШЕНА")
print(f"Розмірність Train: {X_train_full.shape}")
print("="*40)

2. Математичне ядро 3.0 (Стабільне оцінювання та XAI)

In [ ]:
import numpy as np
from scipy.linalg import polar, svd, orthogonal_procrustes

def algorithm_1_stable(X, Xp, lam_equiv=0.1, lam_ortho=1.0, r=3, tau=1e-4):
    """ Оцінювання афінного оператора, стабільне до кількості прикладів """
    # Перетворюємо в (m, n), де m - приклади, n - фічі
    X = np.atleast_2d(X)
    Xp = np.atleast_2d(Xp)
    if X.shape[1] != 768: # Якщо раптом прийшло (768, m)
        X, Xp = X.T, Xp.T

    m, n = X.shape # m - кількість прикладів у кластері, n = 768

    # 1. Центрування (по рядках/прикладах)
    mu_x = np.mean(X, axis=0, keepdims=True)
    mu_xp = np.mean(Xp, axis=0, keepdims=True)
    Xc, Xpc = X - mu_x, Xp - mu_xp

    # 2. Емпіричні генератори J через PCA
    # PCA навчаємо на деформаціях (m, n)
    pca = PCA(n_components=min(r, m-1))
    pca.fit(Xpc - Xc)

    mu_x_flat = mu_x.flatten()
    mu_norm_sq = np.sum(mu_x_flat**2) + 1e-9
    generators = [np.outer(p, mu_x_flat) / mu_norm_sq for p in pca.components_]

    # 3. Формування системи нормальних рівнянь (n x n)
    LHS = Xc.T @ Xc
    RHS = Xpc.T @ Xc

    for J in generators:
        LHS += lam_equiv * (J.T @ J)

    # Ортогональна стабілізація
    R_prior, _ = orthogonal_procrustes(Xc, Xpc)
    LHS += lam_ortho * np.eye(n)
    RHS += lam_ortho * R_prior.T

    # Мікро-зсув для збіжності SVD
    LHS += 1e-7 * np.eye(n)

    # 4. SVD розв'язання
    U, S, Vh = svd(LHS, full_matrices=False)
    S_inv = np.zeros_like(S)
    mask = S > (tau * np.max(S))
    S_inv[mask] = 1.0 / S[mask]

    # Обчислюємо A (n x n)
    # Формула: A^T = inv(LHS) @ RHS^T  => A = RHS @ inv(LHS)
    A = RHS @ (Vh.T * S_inv) @ U.T

    # 5. Реконструкція вектора зсуву t (1, n)
    t = (mu_xp - (mu_x @ A.T)).flatten()

    return A, t

def algorithm_2_profile_stable(A, t):
    """
    Алгоритм 2: Стабільне XAI-профілювання.
    """
    n = A.shape[0]

    # 1. Полярний розклад A = RS
    # Оскільки ми додали ortho-reg, розклад буде стабільним
    try:
        R, S_mat = polar(A)
    except:
        U, s, Vh = svd(A)
        R = U @ Vh
        S_mat = Vh.T @ np.diag(s) @ Vh

    # 2. Кут структурної зміни (формула 25)
    tr_R = np.trace(R)
    cos_theta = np.clip((tr_R - n + 2) / 2, -1.0, 1.0)
    theta = np.degrees(np.arccos(cos_theta))

    # 3. Індекс деформації Def (формула 26)
    _, sigmas, _ = svd(A)
    Def = np.mean(np.abs(sigmas - 1))

    # 4. Контекстуальний зсув
    Shift = np.linalg.norm(t)

    return {"theta": theta, "Def": Def, "Shift": Shift, "det": np.linalg.det(A)}

def predict_hybrid_score(X, Xp, A, t):
    """
    Гібридна метрика: Косинусна подібність після афінного вирівнювання.
    """
    # x_pred = Ax + t
    X_pred = (X @ A.T) + t
    # Нормалізація
    norms = np.linalg.norm(X_pred, axis=1, keepdims=True)
    X_pred_norm = X_pred / (norms + 1e-9)

    # Скалярний добуток (косинус)
    sims = np.sum(X_pred_norm * Xp, axis=1)
    return sims

3. Сценарій В — Пошук оптимального компромісу та XAI-стабільності.
Пошук λ та r. Саме тут визначаються FINAL_A та FINAL_t.

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

# 1. ОБЧИСЛЕННЯ БЕЙСЛАЙНУ (Standard SBERT Cosine)
# Оскільки вектори нормалізовані, косинусна подібність — це скалярний добуток
cos_sims_dev = np.sum(X_dev * Xp_dev, axis=1)
auc_baseline = roc_auc_score(df_dev['Label'], cos_sims_dev)

# 2. ПІДГОТОВКА ДАНИХ ДЛЯ НАВЧАННЯ (Тільки справжні парафрази)
train_indices = df_train[df_train['Label'] == 1].index
X_tr_samples = X_train_full[train_indices]
Xp_tr_samples = Xp_train_full[train_indices]

# 3. ВИЗНАЧЕННЯ СІТКИ ГІПЕРПАРАМЕТРІВ
lambdas_ortho = [100.0, 500.0, 1000.0, 5000.0]  # Контроль жорсткості орієнтації
lambdas_equiv = [0.1, 1.0]                      # Вага генераторів Лі
rs = [3, 5]                                     # Кількість PCA-компонент
tau_val = 1e-3                                  # Поріг фільтрації шуму

results_log = []

print(f"БЕЙСЛАЙН (Cosine Similarity) AUC: {auc_baseline:.4f}")
print("="*75)
print(f"{'λ_ortho':>10} | {'r':>3} | {'λ_equiv':>8} | {'AUC':>8} | {'θ, град.':>10} | {'Def':>8}")
print("-" * 75)

start_time = time.time()

# 4. ЦИКЛ ПЕРЕБОРУ (GRID SEARCH)
for r_val in rs:
    for le in lambdas_equiv:
        for lo in lambdas_ortho:
            try:
                # А. Оцінювання оператора (Алгоритм 1 v3.0)
                A, t = algorithm_1_stable(X_tr_samples, Xp_tr_samples,
                                         lam_equiv=le, lam_ortho=lo, r=r_val, tau=tau_val)

                # Б. Обчислення якості (AUC) через гібридну метрику на Dev
                hybrid_sims = predict_hybrid_score(X_dev, Xp_dev, A, t)
                auc_score = roc_auc_score(df_dev['Label'], hybrid_sims)

                # В. Розрахунок XAI-метрик (Реальний кут та Деформація)
                # Використовуємо підвибірку для стабільного кута трансформації
                X_sample = X_tr_samples[:500]
                X_pred = (X_sample @ A.T) + t
                X_pred_n = X_pred / (np.linalg.norm(X_pred, axis=1, keepdims=True) + 1e-9)
                cos_vals = np.clip(np.sum(X_sample * X_pred_n, axis=1), -1.0, 1.0)
                actual_theta = np.mean(np.degrees(np.arccos(cos_vals)))

                profile = algorithm_2_profile_stable(A, t)
                deformation = profile['Def']

                # Г. Запис результатів
                results_log.append({
                    'lo': lo, 'le': le, 'r': r_val,
                    'AUC': auc_score, 'Theta': actual_theta, 'Def': deformation, 'det': profile['det']
                })

                print(f"{lo:10.1f} | {r_val:3d} | {le:8.2f} | {auc_score:8.4f} | {actual_theta:9.2f}° | {deformation:8.5f}")

            except Exception as e:
                continue

# 5. АВТОМАТИЧНИЙ ВИБІР ТА ФІНАЛІЗАЦІЯ
df_results = pd.DataFrame(results_log)
best_row = df_results.loc[df_results['AUC'].idxmax()]

# Збереження найкращих параметрів
FINAL_LO = best_row['lo']
FINAL_LE = best_row['le']
FINAL_R = int(best_row['r'])
FINAL_A, FINAL_t = algorithm_1_stable(X_tr_samples, Xp_tr_samples, lam_equiv=FINAL_LE, lam_ortho=FINAL_LO, r=FINAL_R)

# 6. ВІЗУАЛІЗАЦІЯ (Двопанельний графік для статті)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# ГРАФІК 1: AUC Trend (Масштабований)
sns.lineplot(data=df_results, x='lo', y='AUC', hue='r', style='le', marker='o', ax=ax1, palette='viridis')
ax1.set_ylim(df_results['AUC'].min() - 0.001, df_results['AUC'].max() + 0.001)
ax1.set_xscale('log')
ax1.set_title("Якість класифікації (Zoomed AUC)")
ax1.set_xlabel("λ_ortho (log scale)")
ax1.set_ylabel("ROC-AUC Score")
ax1.grid(True, which="both", ls="-", alpha=0.3)

# ГРАФІК 2: Deformation vs Stability (XAI Criterion)
sns.lineplot(data=df_results, x='lo', y='Def', hue='r', ax=ax2, palette='magma', marker='s')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_title("Геометрична стабільність (Deformation Index)")
ax2.set_xlabel("λ_ortho (log scale)")
ax2.set_ylabel("Deformation (lower is better)")
ax2.grid(True, which="both", ls="-", alpha=0.3)

plt.suptitle(f"Аналіз Trade-off: Оптимізація афінного оператора (Baseline AUC: {auc_baseline:.4f})", fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('tradeoff_analysis.png', dpi=300)
plt.show()

# 7. СТАТИСТИЧНИЙ ПІДСУМОК ДЛЯ РОЗДІЛУ 4
print("\n" + "="*60)
print("ФІНАЛЬНІ ДАНІ ДЛЯ ПУБЛІКАЦІЇ:")
print(f"Найкраща конфігурація: λ_ortho={FINAL_LO}, λ_equiv={FINAL_LE}, r={FINAL_R}")
print(f"Максимальний AUC:     {best_row['AUC']:.4f}")
print(f"Геометричний інваріант θ: {best_row['Theta']:.2f}° (STD: {df_results['Theta'].std():.4f}°)")
print(f"Мінімальна деформація Def: {best_row['Def']:.6f}")
print(f"Детермінант (det A):      {best_row['det']:.2e}")
print("="*60)

4. Задача 1 — Структурний аналіз оператора (A,t)
Цей код візуалізує «природу» знайденої матриці A та вектора зсуву t. Ми покажемо, як виглядає спектр власних значень (що доводить стабільність) та розподіл елементів зсуву.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Отримання фінальних параметрів
A_final, t_final = algorithm_1_stable(X_tr_samples, Xp_tr_samples,
                                     lam_equiv=FINAL_LE, lam_ortho=FINAL_LO, r=FINAL_R)

# 2. ВІЗУАЛІЗАЦІЯ МАТРИЦІ A (фрагмент 50x50 для наочності)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.heatmap(A_final[:50, :50], cmap='RdBu_r', center=0)
plt.title("Структура матриці A (фрагмент 50x50)")
plt.xlabel("Виміри ембединга")
plt.ylabel("Виміри ембединга")

# 3. СПЕКТРАЛЬНИЙ АНАЛІЗ (Сингулярні значення)
_, sigmas, _ = svd(A_final)
plt.subplot(1, 3, 2)
plt.plot(sigmas, color='blue', linewidth=2)
plt.axhline(1.0, color='red', linestyle='--')
plt.title("Спектр сингулярних значень A")
plt.xlabel("Індекс")
plt.ylabel("Значення σ")
plt.grid(True, alpha=0.3)

# 4. РОЗПОДІЛ ВЕКТОРА ЗСУВУ t
plt.subplot(1, 3, 3)
sns.histplot(t_final, kde=True, color='green')
plt.title("Розподіл елементів вектора t")
plt.xlabel("Значення зсуву")
plt.ylabel("Частота")

plt.tight_layout()
plt.savefig('task1_operator_analysis.png', dpi=300)
plt.show()

print(f"Норма матриці A (Frobenius): {np.linalg.norm(A_final, 'fro'):.4f}")
print(f"Норма вектора зсуву ||t||: {np.linalg.norm(t_final):.4f}")
print(f"Ранг матриці A: {np.linalg.matrix_rank(A_final)}")

5: Задача 2 — Класифікація та ROC-AUC аналіз.
Цей код будує головні доказові графіки: ROC-криву (модель vs бейслайн) та гістограми розподілу похибок, які показують, як саме ми розділяємо парафрази від не-парафраз.
(Сценарій А (Global — Dev). Валідація глобальної моделі на нових темах. Результат AUC = 0.7713)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# --- ДОДОТКОВА ФУНКЦІЯ (якщо вона зникла з пам'яті) ---
def predict_error_local(X, Xp, A, t):
    """ Обчислення норми похибки ||x' - (Ax + t)|| """
    X_pred = (X @ A.T) + t
    return np.linalg.norm(Xp - X_pred, axis=1)

# 1. ВИКОРИСТАННЯ ФІНАЛЬНИХ ПАРАМЕТРІВ (з Клітинки 3)
# Переконуємося, що беремо правильні змінні
try:
    current_A = FINAL_A
    current_t = FINAL_t
except NameError:
    print("Помилка: Виконайте спочатку Клітинку 3!")

# 2. ОТРИМАННЯ СКОРІВ
# Бейслайн (Косинусна подібність)
sims_baseline = np.sum(X_dev * Xp_dev, axis=1)

# Наша модель (Гібридна подібність)
sims_model = predict_hybrid_score(X_dev, Xp_dev, current_A, current_t)

# 3. ПОБУДОВА ROC-КРИВИХ
fpr_b, tpr_b, _ = roc_curve(df_dev['Label'], sims_baseline)
fpr_m, tpr_m, _ = roc_curve(df_dev['Label'], sims_model)

auc_b = auc(fpr_b, tpr_b)
auc_m = auc(fpr_m, tpr_m)

plt.figure(figsize=(16, 6))

# ГРАФІК 1: ROC-AUC
plt.subplot(1, 2, 1)
plt.plot(fpr_b, tpr_b, color='gray', linestyle='--', label=f'SBERT Baseline (AUC = {auc_b:.4f})')
plt.plot(fpr_m, tpr_m, color='darkorange', linewidth=3, label=f'Affine Operator (AUC = {auc_m:.4f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle=':')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Порівняння ROC-кривих (Classification Performance)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# ГРАФІК 2: Розподіл похибок наближення (XAI)
# Рахуємо похибку ||x' - T(x)|| для Label 0 та 1
errors_dev = predict_error_local(X_dev, Xp_dev, current_A, current_t)
df_err = pd.DataFrame({
    'Error': errors_dev,
    'Label': df_dev['Label'].map({1: 'Paraphrase (P)', 0: 'Non-Paraphrase (NP)'})
})

plt.subplot(1, 2, 2)
sns.kdeplot(data=df_err, x='Error', hue='Label', fill=True, common_norm=False, palette='Set1')
plt.title('Розподіл похибок афінного наближення (XAI Verification)')
plt.xlabel('Residual Error ||x\' - T(x)||')
plt.ylabel('Щільність розподілу')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('task2_classification_analysis.png', dpi=300)
plt.show()

print(f"Результат аналізу:")
print(f"1. AUC Бейслайну: {auc_b:.4f}")
print(f"2. AUC Афінної моделі: {auc_m:.4f}")
print(f"3. Відносний успіх: {(auc_m/auc_b)*100:.2f}% від потужності нелінійної моделі.")

6: XAI-Аналіз — Спектр геометричної складності (XAI-аналіз (Геометричний коридор))

In [ ]:
# 1. Розрахунок кутів для ВСІХ справжніх парафраз у Dev-сеті
p_mask = (df_dev['Label'] == 1)
X_p = X_dev[p_mask]
Xp_p = Xp_dev[p_mask]
df_p = df_dev[p_mask].copy()

# Обчислюємо кути трансформації
dot_p = np.clip(np.sum(X_p * Xp_p, axis=1), -1.0, 1.0)
df_p['theta_val'] = np.degrees(np.arccos(dot_p))

# 2. ВІЗУАЛІЗАЦІЯ: Гістограма "Геометричного коридору"
plt.figure(figsize=(10, 6))
sns.histplot(df_p['theta_val'], kde=True, color='purple', bins=30)
plt.axvline(df_p['theta_val'].mean(), color='red', linestyle='--', label=f"Середній кут: {df_p['theta_val'].mean():.2f}°")
plt.title("Розподіл кутів трансформації для справжніх парафраз (Geometric Corridor)")
plt.xlabel("Кут повороту θ, градуси")
plt.ylabel("Кількість пар")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('angular_distribution.png', dpi=300)
plt.show()

# 3. ВИБІР ЕКСТРЕМАЛЬНИХ ПРИКЛАДІВ
# Найменші кути (Лексична близькість)
simple_cases = df_p.sort_values('theta_val').head(3)
# Найбільші кути (Синтаксична перебудова)
complex_cases = df_p.sort_values('theta_val').tail(3)

print("\nТАБЛИЦЯ 3. СПЕКТР ГЕОМЕТРИЧНОЇ СКЛАДНОСТІ ПАРАФРАЗУВАННЯ")
spectrum_results = []

for label, cases in [("Прості (Лексичні)", simple_cases), ("Складні (Структурні)", complex_cases)]:
    for _, row in cases.iterrows():
        spectrum_results.append({
            'Тип трансформації': label,
            'Текст 1 (Оригінал)': row['Sent1_raw'][:70] + "...",
            'Текст 2 (Парафраз)': row['Sent2_clean'][:70] + "...",
            'Кут θ': f"{row['theta_val']:.2f}°"
        })

df_spectrum = pd.DataFrame(spectrum_results)
display(df_spectrum)

7. Код для реалізації сценаріїв: Сценарій Б (Local — Dev): Локальні моделі на нових темах; Сценарій А.1 (Global — Train): Глобальна модель на відомих темах; Сценарій Б.1 (Local — Train): Локальні моделі на відомих темах ("теоретична межа")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score

# --- ПАРАМЕТРИ З ТАБЛИЦІ 1 ---
L_EQUIV = 1.0
L_ORTHO = 5000.0
R_COMP = 5
N_CLUSTERS = 15

print(f"Реалізація Сценаріїв Б, Б.1 та А.1 (Кластеризація K={N_CLUSTERS})...")

# 1. КЛАСТЕРИЗАЦІЯ (на основі всіх даних Train)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
all_clusters_train = kmeans.fit_predict(X_train_full)
all_clusters_dev = kmeans.predict(X_dev)

# 2. НАВЧАННЯ ЛОКАЛЬНИХ МОДЕЛЕЙ (на парафразах з кожного кластера)
local_models = {}
for c in range(N_CLUSTERS):
    # Беремо тільки парафрази (Label=1) всередині кластера
    mask = (all_clusters_train == c) & (df_train['Label'] == 1)
    if mask.sum() > 20:
        A_c, t_c = algorithm_1_stable(X_train_full[mask], Xp_train_full[mask],
                                     lam_equiv=L_EQUIV, lam_ortho=L_ORTHO, r=R_COMP)
        local_models[c] = (A_c, t_c)

# 3. ФУНКЦІЯ ДЛЯ ТЕСТУВАННЯ СЦЕНАРІЇВ
def evaluate_scenarios(X_set, Xp_set, labels, cluster_assignments, use_local=True):
    scores = []
    for i in range(len(X_set)):
        c_idx = cluster_assignments[i]
        # Вибір моделі
        if use_local and c_idx in local_models:
            A, t = local_models[c_idx]
        else:
            A, t = FINAL_A, FINAL_t # Глобальна модель як fallback

        # Гібридна подібність
        x_pred = (X_set[i].reshape(1,-1) @ A.T) + t
        x_pred_n = x_pred / (np.linalg.norm(x_pred) + 1e-9)
        scores.append(np.dot(x_pred_n.flatten(), Xp_set[i]))
    return roc_auc_score(labels, scores)

# --- ОБЧИСЛЕННЯ УСІХ AUC ДЛЯ ТАБЛИЦІ 4 ---

# Сценарій А: Global - Dev (вже був, перерахуємо для контролю)
auc_a = evaluate_scenarios(X_dev, Xp_dev, df_dev['Label'], all_clusters_dev, use_local=False)

# Сценарій Б: Local - Dev
auc_b = evaluate_scenarios(X_dev, Xp_dev, df_dev['Label'], all_clusters_dev, use_local=True)

# Сценарій А.1: Global - Train
auc_a1 = evaluate_scenarios(X_train_full, Xp_train_full, df_train['Label'], all_clusters_train, use_local=False)

# Сценарій Б.1: Local - Train
auc_b1 = evaluate_scenarios(X_train_full, Xp_train_full, df_train['Label'], all_clusters_train, use_local=True)

# --- ВИВІД ПІДСУМКОВОЇ ТАБЛИЦІ 4 ---
data4 = {
    "Сценарій": ["A (Global)", "Б (Local)", "Б.1 (Local)", "A.1 (Global)"],
    "Вибірка": ["Dev (Тест)", "Dev (Тест)", "Train (Навчальна)", "Train (Навчальна)"],
    "Тематична новизна": ["Нові теми", "Нові теми", "Відомі теми", "Відомі теми"],
    "AUC": [auc_a, auc_b, auc_b1, auc_a1],
    "Відносна точність до SBERT (%)": [auc_a/0.8405*100, auc_b/0.8405*100, auc_b1/0.8405*100, auc_a1/0.8405*100]
}

df_table_4 = pd.DataFrame(data4)
print("\nТАБЛИЦЯ 4. ПОРІВНЯЛЬНА ЕФЕКТИВНІСТЬ АФІННОЇ МОДЕЛІ")
print(df_table_4.to_string(index=False))

8. Візуалізація афінної трубки (3D Affine Tube)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import KMeans

def plot_affine_tube(X_set, Xp_set, cluster_labels, A, t, title="Affine Tube"):
    # 1. Прогноз та похибка
    X_pred = (X_set @ A.T) + t
    X_pred_n = X_pred / (np.linalg.norm(X_pred, axis=1, keepdims=True) + 1e-9)

    # Радіус (R) = Похибка наближення
    errors = np.linalg.norm(Xp_set - X_pred_n, axis=1)

    # Кут (Phi) = Кут між оригіналом та парафразом
    dot_p = np.clip(np.sum(X_set * Xp_set, axis=1), -1.0, 1.0)
    angles_rad = np.arccos(dot_p)

    # Висота (Z) = Косинусна подібність
    z_vals = dot_p

    # Перехід до декартових координат для 3D циліндра
    x_coords = errors * np.cos(angles_rad)
    y_coords = errors * np.sin(angles_rad)
    z_coords = z_vals

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    # 2. Малювання афінної трубки (напівпрозорий циліндр)
    tube_radius = np.percentile(errors, 80) # Межа 80% парафраз
    z_range = np.linspace(z_coords.min(), z_coords.max(), 20)
    phi_range = np.linspace(0, 2*np.pi, 30)
    phi_grid, z_grid = np.meshgrid(phi_range, z_range)
    cx = tube_radius * np.cos(phi_grid)
    cy = tube_radius * np.sin(phi_grid)
    ax.plot_surface(cx, cy, z_grid, alpha=0.07, color='blue')

    # 3. Центральна вісь
    ax.plot([0, 0], [0, 0], [z_coords.min(), z_coords.max()], color='red', linestyle='--', linewidth=2, label="Центральна вісь T(x)")

    # 4. Точки (колір за кластером)
    scatter = ax.scatter(x_coords, y_coords, z_coords, c=cluster_labels, cmap='tab20', s=12, alpha=0.7)

    ax.set_xlabel('Відхилення (X)')
    ax.set_ylabel('Відхилення (Y)')
    ax.set_zlabel('Семантична подібність (Cosine)')
    ax.set_title(title, fontsize=14)

    ax.view_init(elev=25, azim=45)
    plt.tight_layout()
    plt.show()

# --- ПЕРЕВІРКА ТА ЗАПУСК ---

# Якщо кластери ще не створені - створимо їх
if 'all_clusters_train' not in locals():
    print("Кластеризація не знайдена. Виконується швидкий K-Means...")
    kmeans_quick = KMeans(n_clusters=10, random_state=42, n_init=5)
    all_clusters_train = kmeans_quick.fit_predict(X_train_full)
    all_clusters_dev = kmeans_quick.predict(X_dev)

# Вибірка парафраз для візуалізації
idx_tr_p = np.where(df_train['Label'].values[train_indices] == 1)[0]
# Обмежимо кількість точок для швидкості (1000 точок)
vis_idx = np.random.choice(idx_tr_p, min(1000, len(idx_tr_p)), replace=False)

print("Малювання 3D Affine Tube для Train...")
plot_affine_tube(X_tr_samples[vis_idx], Xp_tr_samples[vis_idx],
                all_clusters_train[train_indices][vis_idx],
                FINAL_A, FINAL_t, title="Affine Paraphrase Tube (Train Set)")

print("Малювання 3D Affine Tube для Dev...")
dev_p_mask = (df_dev['Label'] == 1)
# Беремо всі парафрази з Dev
plot_affine_tube(X_dev[dev_p_mask], Xp_dev[dev_p_mask],
                all_clusters_dev[dev_p_mask],
                FINAL_A, FINAL_t, title="Affine Paraphrase Tube (Dev Set)")

9. 2D-візуалізація семантичного коридору (Affine Tube Projection)

In [ ]:
# Клітинка 9: 2D-візуалізація семантичного коридору
def plot_2d_corridor(X_set, Xp_set, cluster_labels, A, t, title="2D Projection"):
    X_pred = (X_set @ A.T) + t
    X_pred_n = X_pred / (np.linalg.norm(X_pred, axis=1, keepdims=True) + 1e-9)

    # Y: Похибка (Відхилення від осі)
    errors = np.linalg.norm(Xp_set - X_pred_n, axis=1)

    # X: Кут трансформації
    dot_p = np.clip(np.sum(X_set * Xp_set, axis=1), -1.0, 1.0)
    angles = np.degrees(np.arccos(dot_p))

    plt.figure(figsize=(12, 7))

    # Малювання точок
    scatter = plt.scatter(angles, errors, c=cluster_labels, cmap='tab20', alpha=0.5, s=20)

    # Визначення межі (90-й перцентиль)
    boundary = np.percentile(errors, 90)
    plt.axhline(y=boundary, color='red', linestyle='--', label=f'Межа допуску (90%): {boundary:.3f}')

    # Оформлення
    plt.fill_between([angles.min(), angles.max()], 0, boundary, color='gray', alpha=0.1)
    plt.xlabel('Кут структурної трансформації θ, градуси')
    plt.ylabel('Відхилення від ідеальної траєкторії (Error)')
    plt.title(title)
    plt.colorbar(scatter, label='Індекс кластера')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

print("Побудова 2D-проекції для Dev...")
dev_p_mask = (df_dev['Label'] == 1)
plot_2d_corridor(X_dev[dev_p_mask], Xp_dev[dev_p_mask],
                all_clusters_dev[dev_p_mask],
                FINAL_A, FINAL_t, title="Semantic Corridor Boundary (Dev Set)")

10. Розширений XAI-Аудит: Лінгвістична інтерпретація через локальні геометричні профілі

In [ ]:
# 10: Розширений XAI-Аудит: Лінгвістична інтерпретація через локальні геометричні профілі

import pandas as pd
import numpy as np
from scipy.linalg import polar, svd
from IPython.display import display, HTML

print("Формування комплексної таблиці XAI-профілювання (на основі локальних операторів)...\n")

# 1. Створюємо реєстр геометричних профілів для кожного кластера (теми)
cluster_profiles = {}

for c_idx, (A_c, t_c) in local_models.items():
    n = A_c.shape[0]

    # Полярний розклад для локальної матриці
    try:
        R, _ = polar(A_c)
    except:
        U, s, Vh = svd(A_c)
        R = U @ Vh

    # ВИПРАВЛЕНА ФОРМУЛА КУТА ДЛЯ ВИСОКОМІРНИХ ПРОСТОРІВ (n=768)
    tr_R = np.trace(R)
    cos_theta = np.clip(tr_R / n, -1.0, 1.0)
    theta = np.degrees(np.arccos(cos_theta))

    _, sigmas, _ = svd(A_c)
    Def = np.mean(np.abs(sigmas - 1))
    Shift = np.linalg.norm(t_c)
    det_A = np.linalg.det(A_c)

    cluster_profiles[c_idx] = {
        'Theta': theta, 'Def': Def, 'Shift': Shift, 'Det': det_A, 'A': A_c, 't': t_c
    }

# 2. Розраховуємо показники для справжніх парафраз (Label=1) у Dev-вибірці
xai_data = []

for i in range(len(df_dev)):
    if df_dev['Label'].iloc[i] == 1:
        c_idx = all_clusters_dev[i]
        if c_idx in cluster_profiles: # Якщо для теми є локальна модель
            prof = cluster_profiles[c_idx]
            x = X_dev[i]
            xp = Xp_dev[i]

            # Обчислюємо похибку наближення для конкретної пари
            x_pred = (x.reshape(1, -1) @ prof['A'].T) + prof['t']
            x_pred_n = x_pred / (np.linalg.norm(x_pred) + 1e-9)
            err = np.linalg.norm(xp - x_pred_n.flatten())

            xai_data.append({
                'Index': i,
                'Text 1': df_dev['Sent1_clean'].iloc[i][:70] + "...",
                'Text 2': df_dev['Sent2_clean'].iloc[i][:70] + "...",
                'Error': err,
                'Theta': prof['Theta'],
                'Def': prof['Def'],
                'Shift': prof['Shift'],
                'Det': prof['Det']
            })

df_xai = pd.DataFrame(xai_data)

# 3. Відбираємо 4 екстремальні XAI-кейси
results =[]

def add_case(category, row, explanation):
    results.append({
        'Категорія (Тип)': category,
        'Текст 1 (Оригінал)': row['Text 1'],
        'Текст 2 (Парафраз)': row['Text 2'],
        'Похибка (Error)': f"{row['Error']:.3f}",  # ДОДАНО КОЛОНКУ ПОХИБКИ
        'Кут θ': f"{row['Theta']:.2f}°",           # ВИПРАВЛЕНО ФОРМАТ КУТА
        'Деформація (Def)': f"{row['Def']:.6f}",   # ЗБІЛЬШЕНО ТОЧНІСТЬ ДО 6 ЗНАКІВ
        'Зсув (Shift)': f"{row['Shift']:.2f}",
        'Хіральність (Det)': "Від'ємна" if row['Det'] < 0 else "Додатна",
        'XAI Інтерпретація': explanation
    })

# Кейс 1: Ідеальна ізометрія (Мінімальна деформація Def)
case1 = df_xai.loc[df_xai['Def'].idxmin()]
add_case('Ізометричний парафраз', case1,
         "Деформація майже нульова. Зберігається 100% інформаційного обсягу. Відбулася лише структурна перестановка слів (ротація θ).")

# Кейс 2: Семантична хіральність (Від'ємний детермінант)
try:
    # Шукаємо від'ємний детермінант
    chiral_cases = df_xai[df_xai['Det'] < 0]
    if not chiral_cases.empty:
        case2 = chiral_cases.iloc[0]
    else:
        # Якщо від'ємних немає, беремо найменший (для відмовостійкості коду)
        case2 = df_xai.loc[df_xai['Det'].idxmin()]
    add_case('Хіральна інверсія', case2,
             "Детермінант < 0 вказує на дзеркальне відображення простору. Лінгвістично це означає глибоку інверсію сенсу: зміну логічного наголосу або перехід з активу в пасив.")
except IndexError: pass

# Кейс 3: Прагматичний зсув (Максимальний Shift)
case3 = df_xai.loc[df_xai['Shift'].idxmax()]
add_case('Контекстуальний зсув', case3,
         "Великий вектор трансляції (Shift). Структура збережена, але семантичне поле 'перенесено' через зміну тону, емоційного забарвлення або додавання контексту.")

# Кейс 4: Семантична аномалія (Найбільша похибка Error)
case4 = df_xai.loc[df_xai['Error'].idxmax()]
add_case('Аномалія (Помилка)', case4,
         f"Похибка наближення ({case4['Error']:.2f}) виходить за межі афінної трубки. Алгоритм вказує на те, що тексти не є справжніми парафразами через надмірний лексичний шум.")

# 4. Вивід красивої таблиці
df_results = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(HTML(df_results.to_html(index=False, justify='left')))

11. Код для Клітинки №11 (Zero-Shot Аудит Галюцинацій)

In [ ]:
# ==============================================================================
# 11: ZERO-SHOT ВАЛІДАЦІЯ НА HALUEVAL (Cheap Geometric Check)
# Детекція галюцинацій через вихід за межі семантичного коридору
# ==============================================================================

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Допоміжна функція, якщо вона не була збережена в пам'яті з Клітинки 5
def predict_error_local(X, Xp, A, t):
    """ Обчислення норми похибки ||x' - (Ax + t)|| """
    X_pred = (X @ A.T) + t
    return np.linalg.norm(Xp - X_pred, axis=1)

print("Крок 1: Завантаження та підготовка даних HaluEval (qa_data.json)...")
file_path = 'qa_data.json' # Вкажіть шлях до локального файлу

try:
    # Зчитуємо файл порядково, оскільки це формат JSON Lines
    halu_data =[]
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip(): # Перевірка на порожні рядки
                halu_data.append(json.loads(line))

    df_halu = pd.DataFrame(halu_data)

    # Для швидкості візьмемо випадкову вибірку з 1000 пар (можете змінити на len(df_halu) для повного прогону)
    n_samples = min(1000, len(df_halu))
    df_halu = df_halu.sample(n_samples, random_state=42).reset_index(drop=True)
    print(f"Успішно завантажено. Відібрано {n_samples} пар для тестування.")

except FileNotFoundError:
    print(f"ПОМИЛКА: Файл {file_path} не знайдено. Будь ласка, завантажте його.")
    raise

# Крок 2: Векторизація текстів
print("\nКрок 2: Генерація 768D ембедингів для правильних та галюцинаторних відповідей...")
# X - Правильна відповідь (Базовий факт)
X_halu_true = encode_and_normalize(df_halu['right_answer'].tolist())
# Xp - Галюцинаторна відповідь (Аномальний перехід)
Xp_halu_fake = encode_and_normalize(df_halu['hallucinated_answer'].tolist())

# Крок 3: Застосування Cheap Geometric Check (Магістральний оператор)
print("\nКрок 3: Обчислення геометричних метрик (Zero-Shot)...")
# Використовуємо глобальні FINAL_A та FINAL_t, знайдені раніше на PIT-2015
errors_halu = predict_error_local(X_halu_true, Xp_halu_fake, FINAL_A, FINAL_t)

# Додамо результати у DataFrame для аналізу
df_halu['Error'] = errors_halu

# Порогове значення (boundary) з навчання на PIT-2015 (90-й перцентиль)
THRESHOLD = 1.108

# Відсоток успішної детекції галюцинацій (ті, що випали за межі трубки)
detected_hallucinations = (df_halu['Error'] > THRESHOLD).sum()
detection_rate = (detected_hallucinations / n_samples) * 100

print(f"\nРезультат детекції:")
print(f"Виявлено галюцинацій (Error > {THRESHOLD}): {detected_hallucinations} з {n_samples} ({detection_rate:.2f}%)")

# ==============================================================================
# Крок 4: ВІЗУАЛІЗАЦІЯ РОЗПОДІЛУ ПОХИБОК
# ==============================================================================
plt.figure(figsize=(10, 6))
sns.histplot(df_halu['Error'], kde=True, color='crimson', bins=40,
             label='Галюцинації (HaluEval)')

# Додаємо лінію межі допуску з нашої моделі
plt.axvline(x=THRESHOLD, color='navy', linestyle='--', linewidth=2,
            label=f'Межа допуску з PIT-2015 (Error = {THRESHOLD})')

plt.title('Zero-Shot Детекція Галюцинацій: Розподіл геометричної похибки', fontsize=14)
plt.xlabel("Похибка афінного наближення ||x' - (Ax+t)||", fontsize=12)
plt.ylabel('Кількість пар', fontsize=12)

# Зафарбовуємо зону аномалії
plt.axvspan(THRESHOLD, df_halu['Error'].max() + 0.1, color='red', alpha=0.1, label='Зона Галюцинації (Аномалія)')
plt.axvspan(df_halu['Error'].min() - 0.1, THRESHOLD, color='green', alpha=0.1, label='Зона легітимного парафразу')

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('halueval_zero_shot.png', dpi=300)
plt.show()

# ==============================================================================
# Крок 5: ЛІНГВІСТИЧНИЙ АУДИТ ЕКСТРЕМАЛЬНИХ ВИПАДКІВ
# ==============================================================================
print("\nТАБЛИЦЯ: Приклади виявлених галюцинацій (Найбільший вихід за межі многовиду)")

# Відберемо 5 прикладів з найбільшою похибкою
top_hallucinations = df_halu.sort_values(by='Error', ascending=False).head(5)

halu_results =[]
for idx, row in top_hallucinations.iterrows():
    # Розрахунок кута для конкретної пари, щоб показати повний профіль
    # Шукаємо індекс рядка у відфільтрованому масиві векторів
    loc_idx = df_halu.index.get_loc(idx)
    x = X_halu_true[loc_idx]
    xp = Xp_halu_fake[loc_idx]

    dot_p = np.clip(np.sum(x * xp), -1.0, 1.0)
    theta = np.degrees(np.arccos(dot_p))

    halu_results.append({
        'Контекст (Знання)': str(row['knowledge'])[:80] + "...",
        'Факт (Оригінал)': row['right_answer'],
        'Галюцинація (Парафраз)': row['hallucinated_answer'],
        'Похибка (Error)': f"{row['Error']:.3f} 🚨",
        'Кут θ': f"{theta:.2f}°"
    })

df_halu_results = pd.DataFrame(halu_results)
pd.set_option('display.max_colwidth', None)
display(HTML(df_halu_results.to_html(index=False, justify='left')))